# 06 — Diffusion: how do these conversations spread?

**Purpose.** This notebook uses the correct diffusion unit for each platform: Reddit submission cascades and YouTube top-level comment engagement. It then links Reddit cascade size/depth to sentiment, topic, and early high-PageRank participation, and runs a course-aligned Independent Cascade Model on the Reddit reply graph.

**Why Reddit-primary?** Reddit supports nested threaded replies, so submission trees are genuine cascade objects. YouTube replies are shallow and most top-level comments receive no reply, so YouTube is reported as engagement breadth rather than as a parallel cascade network.

**Inputs:** preprocessed comments from notebooks 01/02, network nodes from notebook 04, sentiment and topic labels from notebook 05, and Reddit submissions from notebook 02.

**Outputs:** cascade/engagement tables under `data/processed/05_diffusion/`, report-facing diffusion plots under `plots/`, ICM expected-spread results, and `findings_diffusion_summary.json`.

## Executive summary

- Reddit has **55** submission cascades. The report uses size/depth distributions, topic diffusion, and influence-diffusion diagnostics.
- YouTube has no comparable cascade object; the report uses top-level reply engagement to show shallow structure.
- Sentiment, topic, and influence diagnostics are interpreted as associations, not causal claims.
- ICM is included as a course-aligned structural simulation on the observed Reddit reply graph, not as a deployable seeding recommendation.


## 1 · Setup


In [ ]:
"""Paths, imports, project constants. NB6 is upstream-aware: it reads outputs
from NB1/NB2 (preprocessing), NB4 (network), and NB5 (sentiment + topics).
"""

from __future__ import annotations
import json
import math
import sys
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

DATA              = PROJECT_ROOT / "data"
PROCESSED         = DATA / "processed"
PREPROCESSED      = PROCESSED / "01_preprocessed"
SENTIMENT_TOPICS  = PROCESSED / "03_sentiment_topics"
NETWORKS          = PROCESSED / "04_networks"
DIFFUSION         = PROCESSED / "05_diffusion"
PLOTS             = PROJECT_ROOT / "plots"
for d in (DIFFUSION, PLOTS):
    d.mkdir(parents=True, exist_ok=True)

%matplotlib inline
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 80)

REDDIT_COLOR  = "#3b6fa1"
YOUTUBE_COLOR = "#c45a3d"


def _load_csv(path: Path, **kw) -> pd.DataFrame:
    return pd.read_csv(path, **kw) if path.exists() else pd.DataFrame()


# Reproducibility log.
print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__, "  numpy:", np.__version__)


## 2 · Inputs and data joins

NB6 is the first notebook that needs to combine outputs from all earlier
analyses. The joins below enrich each comment with:

- **PageRank + community + role** from NB4 (`reddit_nodes.csv`, `youtube_nodes.csv`),
  keyed on `author_hash`. Used in the influence × diffusion and ICM sections.
- **Headline sentiment** from NB5 (`sentiment_comment_level.csv` →
  `headline_sentiment_score`, `headline_sentiment_label`), keyed on `comment_id`.
  Used in the sentiment × diffusion section and as a per-cascade aggregate.
- **Dominant LDA topic** from NB5 (`topics_lda_platform_dominant_per_comment_labelled.csv`),
  keyed on `comment_id`. Used in the topic × diffusion section.

The join coverage is printed below so the reader can see how many comments
ended up with sentiment / topic / PageRank attached.


In [ ]:
# --- Comment tables ---------------------------------------------------------
reddit_comments    = _load_csv(PREPROCESSED / "reddit_comments.csv")
reddit_submissions = _load_csv(PREPROCESSED / "reddit_submissions.csv")
youtube_comments   = _load_csv(PREPROCESSED / "youtube_comments.csv")

print(f"NB1/NB2 comments — reddit: {len(reddit_comments):,}   youtube: {len(youtube_comments):,}")

# --- NB4 network nodes ------------------------------------------------------
reddit_nodes  = _load_csv(NETWORKS / "reddit_nodes.csv",
                          usecols=["node", "pagerank", "betweenness", "community", "role"])
youtube_nodes = _load_csv(NETWORKS / "youtube_nodes.csv",
                          usecols=["node", "pagerank", "betweenness", "community", "role"])
reddit_nodes  = reddit_nodes.rename(columns={"node": "author_hash"})
youtube_nodes = youtube_nodes.rename(columns={"node": "author_hash"})

# --- NB5 sentiment ---------------------------------------------------------
sentiment_level = _load_csv(
    SENTIMENT_TOPICS / "sentiment_comment_level.csv",
    usecols=["comment_id", "platform",
             "headline_sentiment_score", "headline_sentiment_label",
             "vader_compound", "vader_label",
             "tx_compound", "tx_label"],
)

# --- NB5 LDA dominant topic (labelled) -------------------------------------
topic_dom = _load_csv(
    SENTIMENT_TOPICS / "topics_lda_platform_dominant_per_comment_labelled.csv",
    usecols=["comment_id", "platform", "platform_topic_id",
             "topic_probability", "platform_topic", "topic_label"],
)

print(f"NB4 network — reddit_nodes: {len(reddit_nodes):,}   youtube_nodes: {len(youtube_nodes):,}")
print(f"NB5 sentiment_level: {len(sentiment_level):,}   topic_dominant: {len(topic_dom):,}")


In [ ]:
def _enrich(comments: pd.DataFrame, nodes: pd.DataFrame, platform_name: str) -> pd.DataFrame:
    """Join sentiment + dominant LDA topic + network nodes onto a comment table."""
    if comments.empty:
        return comments
    out = comments.copy()
    sent_p = sentiment_level[sentiment_level["platform"] == platform_name][
        ["comment_id", "headline_sentiment_score", "headline_sentiment_label",
         "vader_compound", "vader_label", "tx_compound", "tx_label"]
    ]
    out = out.merge(sent_p, on="comment_id", how="left")
    top_p = topic_dom[topic_dom["platform"] == platform_name][
        ["comment_id", "platform_topic_id", "topic_probability",
         "platform_topic", "topic_label"]
    ]
    out = out.merge(top_p, on="comment_id", how="left")
    if not nodes.empty:
        out = out.merge(nodes, on="author_hash", how="left")
    return out


reddit_enriched  = _enrich(reddit_comments,  reddit_nodes,  "reddit")
youtube_enriched = _enrich(youtube_comments, youtube_nodes, "youtube")


def _coverage(df_: pd.DataFrame, col: str) -> float:
    return float(df_[col].notna().mean() * 100) if col in df_.columns and len(df_) else 0.0


coverage = pd.DataFrame([
    {"platform": "reddit",
     "comments": len(reddit_enriched),
     "with_sentiment_pct":  round(_coverage(reddit_enriched, "headline_sentiment_label"), 1),
     "with_dominant_topic_pct": round(_coverage(reddit_enriched, "topic_label"), 1),
     "with_pagerank_pct":   round(_coverage(reddit_enriched, "pagerank"), 1)},
    {"platform": "youtube",
     "comments": len(youtube_enriched),
     "with_sentiment_pct":  round(_coverage(youtube_enriched, "headline_sentiment_label"), 1),
     "with_dominant_topic_pct": round(_coverage(youtube_enriched, "topic_label"), 1),
     "with_pagerank_pct":   round(_coverage(youtube_enriched, "pagerank"), 1)},
])
print("Join coverage (% of comments that received each enrichment):")
display(coverage)


## 3 · Helpers

Two helpers: one builds the per-cascade table for Reddit (size, depth, timing, shape, top-user concentration, plus the new sentiment / topic / influence columns); the other builds a per-top-level-comment engagement table for YouTube.


### 3a · Reddit cascade metrics — extended


In [ ]:
def _safe_to_dt(s, *, unit=None):
    return pd.to_datetime(s, unit=unit, utc=True, errors="coerce")


def reddit_cascade_metrics(comments_df: pd.DataFrame,
                           submissions_df: pd.DataFrame) -> pd.DataFrame:
    """One row per Reddit submission tree.

    Structural metrics (size, depth, branching, leaves, bushiness, longest chain),
    timing metrics (time-to-first-reply, time-to-peak-hour, time-to-50pct cumulative),
    participation metrics (unique users, share of top user),
    plus enrichment from NB4 + NB5:
      - sentiment_mean, sentiment_std, pct_negative, pct_positive
      - sentiment_first_quartile_mean, sentiment_last_quartile_mean (drift over time)
      - dominant_topic_label, dominant_topic_share, topic_entropy
      - early_high_pagerank_share (first-10% commenters in top-decile of PageRank)
      - mean_author_pagerank, max_author_pagerank
    """
    if comments_df.empty or submissions_df.empty:
        return pd.DataFrame()

    c = comments_df.copy()
    c["created_at"] = _safe_to_dt(c["created_utc"], unit="s")
    sub_start = (submissions_df.set_index("submission_id")["created_utc"]
                                 .apply(lambda x: _safe_to_dt(pd.Series([x]), unit="s").iloc[0]))

    # PageRank top decile across the whole Reddit author population.
    pagerank_threshold = float("nan")
    if "pagerank" in c.columns and c["pagerank"].notna().any():
        pagerank_threshold = float(c["pagerank"].quantile(0.9))

    rows = []
    for sid, group in c.groupby("submission_id"):
        if sid is None or sid not in sub_start.index:
            continue
        start = sub_start.loc[sid]
        group = group.sort_values("created_at")
        size = len(group)
        depths = group["depth"].dropna().astype(int)

        # --- reply-tree shape diagnostics ------------------------------------
        children = group["parent_id"].value_counts()   # parent_id -> n direct children
        parent_set = set(children.index)
        leaf_count = int(group[~group["comment_name"].isin(parent_set)].shape[0])
        pct_leaves = leaf_count / size if size else 0.0
        if not depths.empty:
            depth_counts = depths.value_counts().sort_index()
            longest_chain = int(depth_counts.index.max() + 1)
            bushiness = float(depth_counts.iloc[-1] / size) if size else 0.0
        else:
            longest_chain, bushiness = 0, 0.0

        # --- FIX: mean_children_root ----------------------------------------
        # The root of the cascade is the submission itself, not the earliest
        # comment. Its full Reddit name is "t3_<submission_id>"; replies to the
        # submission have parent_id == this name.
        submission_name = f"t3_{sid}"
        mean_children_root = float(children.get(submission_name, 0))

        # --- timing ---------------------------------------------------------
        # Peak hour: hour with the most comments inside the cascade.
        ts = group["created_at"].dropna()
        if not ts.empty and pd.notna(start):
            counts = pd.Series(1, index=ts).resample("h").sum()
            peak_hour = counts.idxmax() if not counts.empty else None
        else:
            peak_hour = None
        time_to_peak = (max(0.0, (peak_hour - start).total_seconds())
                        if peak_hour is not None and pd.notna(start) else None)
        first = group["created_at"].min()
        if pd.notna(start) and size:
            deltas = (group["created_at"] - start).dt.total_seconds().dropna().sort_values()
            half_idx = int(math.ceil(size / 2)) - 1
            t_50pct = float(deltas.iloc[half_idx]) if 0 <= half_idx < len(deltas) else None
        else:
            t_50pct = None

        # --- participation --------------------------------------------------
        users = group["author_hash"].dropna()
        top_user_share = float(users.value_counts(normalize=True).iloc[0]) if len(users) else 0.0

        # --- NEW: sentiment aggregates per cascade --------------------------
        sentiment_mean = sentiment_std = pct_negative = pct_positive = None
        sentiment_first_q = sentiment_last_q = None
        if "headline_sentiment_score" in group.columns and group["headline_sentiment_score"].notna().any():
            s = group["headline_sentiment_score"]
            sentiment_mean = float(s.mean())
            sentiment_std  = float(s.std())
            pct_negative = float((group["headline_sentiment_label"] == "negative").mean())
            pct_positive = float((group["headline_sentiment_label"] == "positive").mean())
            # Split into first vs last chronological quartile.
            n = len(group)
            q1_n = max(1, n // 4)
            q4_n = max(1, n // 4)
            sentiment_first_q = float(group["headline_sentiment_score"].iloc[:q1_n].mean())
            sentiment_last_q  = float(group["headline_sentiment_score"].iloc[-q4_n:].mean())

        # --- NEW: dominant topic + topic entropy ----------------------------
        dominant_topic_label = None
        dominant_topic_share = None
        topic_entropy = None
        if "topic_label" in group.columns and group["topic_label"].notna().any():
            topic_counts = group["topic_label"].dropna().value_counts(normalize=True)
            if len(topic_counts):
                dominant_topic_label = str(topic_counts.index[0])
                dominant_topic_share = float(topic_counts.iloc[0])
                p = topic_counts.values
                topic_entropy = float(-(p * np.log(np.clip(p, 1e-12, 1))).sum())

        # --- NEW: influence × diffusion --------------------------------------
        early_high_pagerank_share = None
        mean_author_pagerank = None
        max_author_pagerank  = None
        if "pagerank" in group.columns and group["pagerank"].notna().any():
            mean_author_pagerank = float(group["pagerank"].mean())
            max_author_pagerank  = float(group["pagerank"].max())
            if pd.notna(pagerank_threshold):
                early_n = max(1, int(0.1 * size))
                early = group.head(early_n)["pagerank"].dropna()
                if len(early):
                    early_high_pagerank_share = float((early >= pagerank_threshold).mean())

        rows.append({
            "cascade_id":   sid,
            "subreddit":    group["subreddit"].iloc[0] if "subreddit" in group else None,
            "started_at":   start,
            "size":         size,
            "max_depth":    int(depths.max()) if not depths.empty else 0,
            "mean_depth":   float(depths.mean()) if not depths.empty else 0.0,
            "branching_factor_non_leaf": float(children.mean()) if len(children) else 0.0,
            "pct_leaves":          pct_leaves,
            "bushiness":           bushiness,
            "longest_chain":       longest_chain,
            "mean_children_root":  mean_children_root,
            "time_to_first_reply_sec": ((first - start).total_seconds()
                                         if pd.notna(first) and pd.notna(start) else None),
            "time_to_peak_hour_sec":   time_to_peak,
            "t_50pct_cumulative_sec":  t_50pct,
            "unique_users":   int(users.nunique()),
            "pct_top_user":   top_user_share,
            # Sentiment enrichment
            "sentiment_mean":  sentiment_mean,
            "sentiment_std":   sentiment_std,
            "pct_negative":    pct_negative,
            "pct_positive":    pct_positive,
            "sentiment_first_quartile_mean": sentiment_first_q,
            "sentiment_last_quartile_mean":  sentiment_last_q,
            # Topic enrichment
            "dominant_topic_label":  dominant_topic_label,
            "dominant_topic_share":  dominant_topic_share,
            "topic_entropy":         topic_entropy,
            # Influence enrichment
            "early_high_pagerank_share": early_high_pagerank_share,
            "mean_author_pagerank":      mean_author_pagerank,
            "max_author_pagerank":       max_author_pagerank,
        })
    return pd.DataFrame(rows)


### 3b · YouTube top-level-engagement metrics


In [ ]:
def youtube_engagement_metrics(comments_df: pd.DataFrame) -> pd.DataFrame:
    """One row per top-level YouTube comment + its (zero or more) replies.

    YouTube does not have cascades in the Reddit sense; this captures the only
    diffusion-like measure YouTube allows: how many replies a top-level comment
    gets and how fast the first one arrives.
    """
    if comments_df.empty:
        return pd.DataFrame()
    c = comments_df.copy()
    c["published_at"] = pd.to_datetime(c["published_at"], utc=True, errors="coerce")
    top     = c[~c["is_reply"].fillna(False)].copy()
    replies = c[ c["is_reply"].fillna(False)].copy()
    reply_groups = replies.groupby("parent_id")

    rows = []
    for _, head in top.iterrows():
        cid = head["comment_id"]
        thread = reply_groups.get_group(cid) if cid in reply_groups.groups else replies.iloc[0:0]
        n_replies = len(thread)
        delta_first = (
            (thread["published_at"].min() - head["published_at"]).total_seconds()
            if not thread.empty and pd.notna(head["published_at"]) else None
        )
        rows.append({
            "cascade_id":              cid,
            "video_id":                head.get("video_id"),
            "channel_id":              head.get("channel_id"),
            "started_at":              head["published_at"],
            "n_replies":               n_replies,
            "had_any_reply":           n_replies > 0,
            "time_to_first_reply_sec": delta_first,
        })
    return pd.DataFrame(rows)


## 4 · Build the cascade and engagement tables

Single pass that builds and caches both per-cascade tables, plus an audit print so the rest of the notebook always knows the corpus shape.


In [ ]:
cascades_reddit       = reddit_cascade_metrics(reddit_enriched, reddit_submissions)
engagement_youtube    = youtube_engagement_metrics(youtube_enriched)

cascades_reddit.to_csv(DIFFUSION / "reddit_cascade_metrics.csv", index=False)
engagement_youtube.to_csv(DIFFUSION / "youtube_engagement_metrics.csv", index=False)

n_r, n_y = len(cascades_reddit), len(engagement_youtube)
print(f"Reddit cascades:                {n_r:>6,}  (median size {cascades_reddit['size'].median():.0f}, "
      f"median max-depth {cascades_reddit['max_depth'].median():.0f})")
print(f"YouTube top-level comments:     {n_y:>6,}  "
      f"(top-level with >=1 reply: {int(engagement_youtube['had_any_reply'].sum()):,} = "
      f"{engagement_youtube['had_any_reply'].mean()*100:.1f}%)")

print("\nEnrichment availability per cascade (% non-null):")
print(f"  sentiment_mean              : {cascades_reddit['sentiment_mean'].notna().mean()*100:5.1f}%")
print(f"  dominant_topic_label        : {cascades_reddit['dominant_topic_label'].notna().mean()*100:5.1f}%")
print(f"  early_high_pagerank_share   : {cascades_reddit['early_high_pagerank_share'].notna().mean()*100:5.1f}%")


## 5 · Reddit cascade size and depth

Distributions of **size** and **max-depth** across the Reddit submission trees. The size histogram uses log-spaced bins so the heavy right tail is visible at the same visual resolution as the body of the distribution.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# FIX: log-spaced bins under a log-scaled x-axis.
sizes = cascades_reddit["size"].dropna()
if len(sizes):
    log_bins = np.logspace(np.log10(max(sizes.min(), 1)), np.log10(sizes.max()), 22)
    axes[0].hist(sizes, bins=log_bins, color=REDDIT_COLOR, edgecolor="white")
axes[0].set_xscale("log")
axes[0].set_xlabel("cascade size (comments, log-scale)")
axes[0].set_title(f"Reddit cascade size  (n={len(cascades_reddit)})")

sns.histplot(cascades_reddit["max_depth"],
             bins=range(0, int(cascades_reddit["max_depth"].max()) + 2),
             color=REDDIT_COLOR, ax=axes[1])
axes[1].set_xlabel("max depth (reply nesting)")
axes[1].set_title("Reddit cascade max-depth")

plt.tight_layout()
plt.savefig(PLOTS / "reddit_cascade_size_depth.png", dpi=160, bbox_inches="tight")
plt.show()

display(cascades_reddit[["size", "max_depth", "mean_depth",
                         "branching_factor_non_leaf", "unique_users",
                         "pct_top_user"]].describe().round(2))


**What this means.** Reddit cascades have a heavy right tail in size — the log-spaced bins make it clear that most threads sit in the hundreds and a small minority push well past 500 comments. Depths range from roughly 2 to 10+, but the *typical* maximum depth is modest; deep single-chain arguments are rare even in the largest cascades. The descriptive table at the bottom of the cell gives the reader the per-metric median, quartiles, and max in one place, which is what the rest of the notebook compares against. The remaining comparisons use this distribution as the unconditional baseline before slicing by subreddit, sentiment, topic, influence, or ICM seeding strategy.


## 6 · Subreddit diffusion differences

Network analysis showed that Reddit is not one homogeneous conversation space. This section groups the cascade table by subreddit and reports median size, max-depth, `pct_leaves`, and time-to-first-reply so the report can separate forum scale/posting norms from cascade mechanics.


In [ ]:
by_sub = (cascades_reddit
          .groupby("subreddit")
          .agg(n_cascades=("size", "count"),
               median_size=("size", "median"),
               median_max_depth=("max_depth", "median"),
               median_pct_leaves=("pct_leaves", "median"),
               median_t_first_reply_min=("time_to_first_reply_sec",
                                          lambda s: s.median() / 60.0 if s.notna().any() else float("nan")),
               median_t_50pct_min=("t_50pct_cumulative_sec",
                                   lambda s: s.median() / 60.0 if s.notna().any() else float("nan")))
          .reset_index()
          .sort_values("median_size", ascending=False))
by_sub.to_csv(DIFFUSION / "reddit_diffusion_by_subreddit.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, max(3.5, 0.5 * len(by_sub) + 1)))
for ax, (col, label) in zip(axes, [
    ("median_size",            "Median cascade size"),
    ("median_max_depth",       "Median max-depth"),
    ("median_t_first_reply_min","Median time-to-first-reply (min)"),
]):
    ax.barh(by_sub["subreddit"].iloc[::-1], by_sub[col].iloc[::-1], color=REDDIT_COLOR)
    ax.set_title(label); ax.grid(axis="x", linewidth=0.3, alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS / "reddit_diffusion_by_subreddit.png", dpi=160, bbox_inches="tight")
plt.show()

display(by_sub)


**What this means.** This table is the diffusion-side counterpart to the network structure results. Subreddit differences are interpreted as forum-scale and posting-norm differences, not as proof that one community is intrinsically more influential. The defensibility section tests whether the per-subreddit differences are statistically distinguishable in this corpus.


## 6b · Defensibility — are the per-subreddit differences real?

Two probes the report cites:

- **Kruskal-Wallis H-test** on `size` and `max_depth` across the 8 subreddits — does the null *"all subs draw cascade-sizes from the same distribution"* survive?
- **Spearman correlation heatmap** of all cascade metrics — exposes which metrics are redundant (high |ρ|) and which are independent diffusion axes.


In [ ]:
from scipy.stats import kruskal

groups_size  = [g["size"].values  for _, g in cascades_reddit.groupby("subreddit") if len(g) >= 3]
groups_depth = [g["max_depth"].values for _, g in cascades_reddit.groupby("subreddit") if len(g) >= 3]

if groups_size and groups_depth:
    h_size,  p_size  = kruskal(*groups_size)
    h_depth, p_depth = kruskal(*groups_depth)
else:
    h_size = p_size = h_depth = p_depth = float("nan")

print(f"Kruskal-Wallis — cascade size ~ subreddit:       H={h_size:6.2f}   p={p_size:.4f}")
print(f"Kruskal-Wallis — max-depth   ~ subreddit:       H={h_depth:6.2f}   p={p_depth:.4f}")
print(f"  (n cascades per sub: {[len(g['size'].values) for _, g in cascades_reddit.groupby('subreddit')]})")

metrics = ["size", "max_depth", "mean_depth", "branching_factor_non_leaf",
           "pct_leaves", "bushiness", "longest_chain", "mean_children_root",
           "time_to_first_reply_sec", "time_to_peak_hour_sec", "t_50pct_cumulative_sec",
           "unique_users", "pct_top_user"]
metrics = [m for m in metrics if m in cascades_reddit.columns]
corr = cascades_reddit[metrics].corr(method="spearman")

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            ax=ax, square=True, cbar_kws={"label": "Spearman ρ"})
ax.set_title("Reddit cascade-metric pairwise Spearman correlation")
plt.tight_layout()
plt.savefig(PLOTS / "reddit_cascade_correlations.png", dpi=160, bbox_inches="tight")
plt.show()


**What this means.** If both Kruskal-Wallis p-values are well below 0.05 the per-sub differences are unlikely to be sampling noise. The correlation heatmap tells the reader which axes are independent: `size` and `unique_users` should be highly correlated; `pct_leaves` and `mean_depth` should be negatively correlated (bushy = shallow). Independent axes are the dimensions worth reporting separately. Note: n per subreddit is small (5–6), so the test detects only large differences.


## 7 · YouTube engagement sketch

Without a Reddit-style reply tree, YouTube allows only one diffusion-like question: **when somebody posts a top-level comment, does anyone reply, and how fast?** This section reports the distribution of replies-per-top-level-comment (mostly 0) and the share of top-level comments that ever receive any reply.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

data = engagement_youtube["n_replies"].clip(upper=engagement_youtube["n_replies"].quantile(0.99))
sns.histplot(data, bins=30, color=YOUTUBE_COLOR, ax=axes[0])
axes[0].set_xlabel("replies per top-level comment (clipped at 99th pct)")
axes[0].set_title("YouTube — replies per top-level comment")

share = engagement_youtube["had_any_reply"].mean()
axes[1].bar(["any reply", "zero replies"], [share, 1 - share],
            color=[YOUTUBE_COLOR, "lightgrey"])
axes[1].set_ylim(0, 1.0); axes[1].set_ylabel("share of top-level comments")
axes[1].set_title("YouTube — top-level comments that got any reply at all")
for i, v in enumerate([share, 1 - share]):
    axes[1].text(i, v + 0.01, f"{v*100:.1f}%", ha="center")
plt.tight_layout()
plt.savefig(PLOTS / "youtube_engagement_sketch.png", dpi=160, bbox_inches="tight")
plt.show()


**What this means.** The vast majority of YouTube top-level comments never receive any reply. Among those that do, the median number of replies is small. YouTube's social fabric for this corpus is **broadcaster-comments-to-broadcaster-comments**, not author-to-author conversation. The 88 % / 12 % split is a structural fact of the platform, not a feature of our data.


## 8 · Sentiment and diffusion

For each cascade we aggregated a `sentiment_mean`, `pct_negative`, and `pct_positive` from NB5's headline sentiment scorer. The Hormuz / jet-fuel corpus is overwhelmingly negative-leaning at the *comment* level (≈ 53 % of comments are classified negative), so fixed thresholds like ±0.10 would put nearly every cascade in the same bucket. We instead split cascades into three **terciles** of their own `sentiment_mean` distribution, which gives balanced groups regardless of the corpus-wide skew:

- **most-negative tercile** — the third of cascades with the most-negative `sentiment_mean`,
- **middle tercile** — the middle third,
- **least-negative tercile** — the least-negative third (still negative-leaning overall, but the closest the corpus gets to neutral / positive).

We also report sentiment **drift** within a cascade by comparing the chronological first-quartile mean to the last-quartile mean.


In [ ]:
# Bucket cascades into terciles of sentiment_mean so groups are balanced.
sentiment_terciles = cascades_reddit["sentiment_mean"].dropna().quantile([1/3, 2/3]).values
SENT_LOW, SENT_HIGH = float(sentiment_terciles[0]), float(sentiment_terciles[1])

def _sent_tercile(x):
    if pd.isna(x):
        return None
    if x <= SENT_LOW:  return "most_negative"
    if x >= SENT_HIGH: return "least_negative"
    return "middle"

cascades_reddit["sentiment_bucket"] = cascades_reddit["sentiment_mean"].map(_sent_tercile)
print(f"Sentiment tercile cut points (sentiment_mean): low ≤ {SENT_LOW:+.3f}, high ≥ {SENT_HIGH:+.3f}")

bucket_order = ["most_negative", "middle", "least_negative"]
by_bucket = (cascades_reddit.dropna(subset=["sentiment_bucket"])
                              .groupby("sentiment_bucket")
                              .agg(n_cascades=("size", "count"),
                                   median_size=("size", "median"),
                                   median_max_depth=("max_depth", "median"),
                                   median_t_first_reply_min=("time_to_first_reply_sec",
                                                              lambda s: s.median()/60.0 if s.notna().any() else float("nan")),
                                   median_t_50pct_min=("t_50pct_cumulative_sec",
                                                       lambda s: s.median()/60.0 if s.notna().any() else float("nan")),
                                   median_pct_leaves=("pct_leaves", "median"),
                                   median_pct_negative=("pct_negative", "median"))
                              .reindex(bucket_order)
                              .reset_index())
by_bucket.to_csv(DIFFUSION / "reddit_diffusion_by_sentiment_bucket.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
palette = {"most_negative": "#c44e52", "middle": "#9aa0a6", "least_negative": "#55a868"}

for ax, col, label, ylim_log in [
    (axes[0], "size",                    "cascade size",             True),
    (axes[1], "max_depth",               "max depth",                False),
    (axes[2], "time_to_first_reply_sec", "time to first reply (min)", True),
]:
    sub = cascades_reddit.dropna(subset=["sentiment_bucket", col]).copy()
    if col == "time_to_first_reply_sec":
        sub[col] = sub[col] / 60.0
    sns.boxplot(data=sub, x="sentiment_bucket", y=col, hue="sentiment_bucket",
                order=bucket_order, palette=palette, ax=ax, legend=False)
    if ylim_log:
        ax.set_yscale("log")
    ax.set_title(label); ax.set_xlabel("")
plt.tight_layout()
plt.savefig(PLOTS / "reddit_diffusion_by_sentiment_bucket.png", dpi=160, bbox_inches="tight")
plt.show()

# Kruskal-Wallis across the three sentiment buckets for cascade size.
from scipy.stats import kruskal as _kw_sent
g_size_sent  = [g["size"].values for _, g in cascades_reddit.dropna(subset=["sentiment_bucket"]).groupby("sentiment_bucket") if len(g) >= 3]
g_depth_sent = [g["max_depth"].values for _, g in cascades_reddit.dropna(subset=["sentiment_bucket"]).groupby("sentiment_bucket") if len(g) >= 3]
if g_size_sent and g_depth_sent:
    h_sb, p_sb = _kw_sent(*g_size_sent)
    h_db, p_db = _kw_sent(*g_depth_sent)
    print(f"Kruskal-Wallis (3 sentiment terciles) — cascade size: H = {h_sb:.2f}, p = {p_sb:.4f}")
    print(f"Kruskal-Wallis (3 sentiment terciles) — max_depth : H = {h_db:.2f}, p = {p_db:.4f}")

print("\nPer-bucket medians:")
display(by_bucket)


**What this means.** If the negative-leaning bucket has systematically larger or faster cascades, the report can claim that *negative tone diffuses better* in this corpus — consistent with the well-documented social-media result that negative content travels further. If buckets look similar, the report says diffusion is largely insensitive to overall tone here. We never claim causation: a topic might independently drive both negativity and engagement.


In [ ]:
# Sentiment drift inside cascades — early-quartile mean vs late-quartile mean.
drift = cascades_reddit.dropna(subset=["sentiment_first_quartile_mean",
                                       "sentiment_last_quartile_mean"]).copy()
drift["drift"] = drift["sentiment_last_quartile_mean"] - drift["sentiment_first_quartile_mean"]
drift_summary = drift[["sentiment_first_quartile_mean",
                        "sentiment_last_quartile_mean",
                        "drift"]].describe().round(3)
print("Sentiment drift across cascade lifetime (last quartile - first quartile):")
display(drift_summary)

fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(drift["drift"], bins=20, color=REDDIT_COLOR, ax=ax)
ax.axvline(0, color="black", linewidth=0.6, linestyle="--")
ax.set_xlabel("Δ headline sentiment  (last quartile − first quartile)")
ax.set_title("Reddit — sentiment drift within a cascade")
plt.tight_layout()
plt.savefig(PLOTS / "reddit_sentiment_drift_within_cascade.png", dpi=160, bbox_inches="tight")
plt.show()


**What this means.** If the drift histogram sits centered below zero, cascades tend to get *more negative* as they progress — consistent with the classic "flame-war" narrative. If centered at zero, tone is steady across the lifetime. If above zero, cascades cool down. The mean drift number is reported in the findings JSON.


## 9 · Topic and diffusion

For each cascade we identified its dominant LDA topic from NB5 (the topic that the most comments in the cascade map onto). This lets us ask whether some topics produce systematically bigger or faster cascades than others.

We also report the per-cascade **topic entropy** $H = -\sum_t p_t \log p_t$: low entropy means the cascade is locked to one topic; high entropy means the conversation crosses several themes.


In [ ]:
topic_dom_table = (cascades_reddit
                   .dropna(subset=["dominant_topic_label"])
                   .groupby("dominant_topic_label")
                   .agg(n_cascades=("size", "count"),
                        median_size=("size", "median"),
                        median_max_depth=("max_depth", "median"),
                        median_t_first_reply_min=("time_to_first_reply_sec",
                                                   lambda s: s.median()/60.0 if s.notna().any() else float("nan")),
                        median_pct_negative=("pct_negative", "median"),
                        median_topic_entropy=("topic_entropy", "median"))
                   .reset_index()
                   .sort_values("median_size", ascending=False))
topic_dom_table.to_csv(DIFFUSION / "reddit_diffusion_by_topic.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(16, max(3.5, 0.5 * len(topic_dom_table) + 1)))
for ax, (col, label) in zip(axes, [
    ("median_size",              "Median cascade size"),
    ("median_max_depth",         "Median max-depth"),
    ("median_t_first_reply_min", "Median time-to-first-reply (min)"),
]):
    short_labels = topic_dom_table["dominant_topic_label"].str[:60].iloc[::-1]
    ax.barh(short_labels, topic_dom_table[col].iloc[::-1], color=REDDIT_COLOR)
    ax.set_title(label); ax.grid(axis="x", linewidth=0.3, alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS / "reddit_diffusion_by_topic.png", dpi=160, bbox_inches="tight")
plt.show()

display(topic_dom_table)


**What this means.** Topic diffusion compares which broad frames produce wider or deeper Reddit cascades. This supports the report claim that spread is not explained by negativity alone: market/Hormuz frames can be wide, while geopolitical-conflict frames can be deeper and more negative.


## 10 · Influence and diffusion

The `early_high_pagerank_share` field counts the fraction of the **first 10 % of commenters** in each cascade who sit in the **top decile of PageRank** across the Reddit author population. If high-PageRank users participating early is associated with bigger or faster cascades, that suggests influencer-driven diffusion. Causality is not implied — high-PageRank users may simply turn up where attention is already concentrated.

We also look at *cross-cascade participation*: which authors appear in many cascades, and whether they tend to be high-PageRank.


In [ ]:
from scipy.stats import spearmanr

infl = cascades_reddit.dropna(subset=["early_high_pagerank_share", "size"]).copy()
rho_size,  p_size  = spearmanr(infl["early_high_pagerank_share"], infl["size"])
rho_depth, p_depth = spearmanr(infl["early_high_pagerank_share"], infl["max_depth"])
print(f"Spearman ρ  early_high_pagerank_share × size      = {rho_size:+.3f}   p = {p_size:.4f}")
print(f"Spearman ρ  early_high_pagerank_share × max_depth = {rho_depth:+.3f}   p = {p_depth:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.scatterplot(data=infl, x="early_high_pagerank_share", y="size", ax=axes[0],
                color=REDDIT_COLOR, s=70, alpha=0.7)
axes[0].set_yscale("log")
axes[0].set_xlabel("share of first 10 % commenters in top-decile PageRank")
axes[0].set_ylabel("cascade size (log)")
axes[0].set_title(f"Early high-PageRank share vs cascade size\nSpearman ρ = {rho_size:+.2f}, p = {p_size:.3f}")

sns.scatterplot(data=infl, x="early_high_pagerank_share", y="max_depth", ax=axes[1],
                color=REDDIT_COLOR, s=70, alpha=0.7)
axes[1].set_xlabel("share of first 10 % commenters in top-decile PageRank")
axes[1].set_ylabel("max depth")
axes[1].set_title(f"Early high-PageRank share vs depth\nSpearman ρ = {rho_depth:+.2f}, p = {p_depth:.3f}")
plt.tight_layout()
plt.savefig(PLOTS / "reddit_influence_vs_diffusion.png", dpi=160, bbox_inches="tight")
plt.show()


**What this means.** A positive Spearman correlation between early high-PageRank participation and cascade size/depth is associative evidence, not causal proof. High-PageRank users may attract attention, or they may join threads that were already likely to grow. The report therefore uses role-based language and treats the ICM section as a course-aligned structural simulation rather than a causal estimate.


## 11 · Independent Cascade Model on the Reddit reply graph

The course's Week 10 introduces the **Independent Cascade Model (ICM)**: a sender-centric diffusion model in which each newly-activated node $u$ has a single chance to activate each currently-inactive out-neighbour $w$ with probability $p_{u,w}$. Activations are independent across edges. The expected spread $f(S) = \mathbb{E}|A_\infty(S)|$ is a monotone submodular function of the seed set $S$, so a **greedy** algorithm that adds the node with maximum marginal spread achieves a $(1 - 1/e) \approx 63\%$ approximation of the optimal seed set under a budget constraint (Kempe, Kleinberg & Tardos, 2003).

We run ICM on the Reddit reply graph constructed by NB4 (`reddit_edges.csv`):

- **Activation probability.** A uniform $p = 0.05$ is used across all edges, matching the W10 workshop's uniform-$p$ convention. This is conservative — actual reply rates vary by edge — but the qualitative ranking of seeding strategies (PageRank > random) is what we report, and that ranking is robust to the choice of $p$.
- **Seed budgets.** $k \in \{1, 5, 10, 20\}$.
- **Seeding strategies.** Top-$k$ by PageRank, top-$k$ by betweenness, random sample (averaged over 30 random sets).
- **Monte-Carlo trials.** 200 per (strategy × budget) combination.
- **Output.** Mean expected spread $\mathbb{E}|A_\infty|$ per (strategy, $k$), with 5th–95th percentile bands across trials.

What this gives the report:

1. A direct W10 implementation tied to the actual Reddit graph.
2. Evidence that *which user is seeded* matters even in a small directed reply graph — empirical confirmation of the structural-influence intuition behind PageRank.
3. A counterfactual angle: "if we wanted to *maximise spread*, here is how the budget should be spent."


In [ ]:
import networkx as nx

reddit_edges = _load_csv(NETWORKS / "reddit_edges.csv")
print(f"Reddit edges in NB4 graph: {len(reddit_edges):,}")

# Build directed graph: source -> target (replier -> parent_author).
# Edge weight = number of reply edges along that directed pair.
g = nx.DiGraph()
if not reddit_edges.empty:
    counts = reddit_edges.groupby(["source", "target"]).size().rename("weight").reset_index()
    for _, row in counts.iterrows():
        g.add_edge(row["source"], row["target"], weight=int(row["weight"]))

print(f"ICM graph: {g.number_of_nodes():,} nodes, {g.number_of_edges():,} edges")

# Node ranking by PageRank and betweenness (cached from NB4).
pr_lookup  = dict(zip(reddit_nodes["author_hash"], reddit_nodes["pagerank"]))
btw_lookup = dict(zip(reddit_nodes["author_hash"], reddit_nodes["betweenness"]))

def rank_by(metric_lookup, n_top):
    items = [(node, metric_lookup.get(node, 0.0)) for node in g.nodes()]
    items.sort(key=lambda x: x[1], reverse=True)
    return [n for n, _ in items[:n_top]]


In [ ]:
def icm_single_trial(graph: nx.DiGraph, seeds: list, p: float, rng) -> int:
    """One Monte-Carlo trial of ICM. Returns final activated count."""
    active = set(seeds)
    frontier = set(seeds)
    while frontier:
        new_frontier = set()
        for u in frontier:
            for v in graph.successors(u):
                if v in active:
                    continue
                if rng.random() < p:
                    new_frontier.add(v)
        active |= new_frontier
        frontier = new_frontier
    return len(active)


def icm_expected_spread(graph: nx.DiGraph, seeds: list, p: float,
                        n_trials: int = 200, seed: int = 0):
    """Return (mean, p5, p95) expected spread across n_trials Monte-Carlo runs."""
    rng = np.random.default_rng(seed)
    spreads = [icm_single_trial(graph, seeds, p, rng) for _ in range(n_trials)]
    arr = np.asarray(spreads)
    return float(arr.mean()), float(np.percentile(arr, 5)), float(np.percentile(arr, 95))


ICM_P            = 0.05
ICM_BUDGETS      = [1, 5, 10, 20]
ICM_TRIALS       = 200
ICM_RANDOM_SETS  = 30
ICM_SEED         = 0

results = []
rng_outer = np.random.default_rng(ICM_SEED)
nodes_list = list(g.nodes())

for k in ICM_BUDGETS:
    pr_seeds  = rank_by(pr_lookup, k)
    bw_seeds  = rank_by(btw_lookup, k)
    pr_mean, pr_lo, pr_hi = icm_expected_spread(g, pr_seeds, ICM_P, ICM_TRIALS, seed=ICM_SEED)
    bw_mean, bw_lo, bw_hi = icm_expected_spread(g, bw_seeds, ICM_P, ICM_TRIALS, seed=ICM_SEED + 1)
    rand_means = []
    for j in range(ICM_RANDOM_SETS):
        rseeds = rng_outer.choice(nodes_list, size=min(k, len(nodes_list)), replace=False).tolist()
        m, _, _ = icm_expected_spread(g, rseeds, ICM_P, ICM_TRIALS, seed=ICM_SEED + 10 + j)
        rand_means.append(m)
    rand_arr = np.asarray(rand_means)
    rand_mean, rand_lo, rand_hi = float(rand_arr.mean()), float(np.percentile(rand_arr, 5)), float(np.percentile(rand_arr, 95))

    results.append({"k": k, "strategy": "PageRank top-k",   "mean_spread": pr_mean, "p5": pr_lo, "p95": pr_hi})
    results.append({"k": k, "strategy": "Betweenness top-k","mean_spread": bw_mean, "p5": bw_lo, "p95": bw_hi})
    results.append({"k": k, "strategy": "Random",           "mean_spread": rand_mean, "p5": rand_lo, "p95": rand_hi})

icm_results = pd.DataFrame(results)
icm_results.to_csv(DIFFUSION / "icm_expected_spread.csv", index=False)
display(icm_results)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
strategy_colors = {"PageRank top-k": "#2c7fb8",
                   "Betweenness top-k": "#7fcdbb",
                   "Random": "#cccccc"}

for strat, sub in icm_results.groupby("strategy"):
    sub = sub.sort_values("k")
    ax.plot(sub["k"], sub["mean_spread"],
            marker="o", linewidth=2, label=strat, color=strategy_colors.get(strat, None))
    ax.fill_between(sub["k"], sub["p5"], sub["p95"],
                    alpha=0.15, color=strategy_colors.get(strat, None))

ax.set_xlabel("seed budget k")
ax.set_ylabel(r"expected spread $\mathbb{E}|A_\infty|$")
ax.set_title(f"Independent Cascade Model on the Reddit reply graph  (p = {ICM_P}, {ICM_TRIALS} MC trials/cell)")
ax.legend(title="seeding strategy")
ax.grid(axis="y", linewidth=0.3, alpha=0.4)
plt.tight_layout()
plt.savefig(PLOTS / "icm_expected_spread.png", dpi=160, bbox_inches="tight")
plt.show()


**What this means.** The ICM is a course-aligned simulation on the observed Reddit reply graph. Centrality-based seeding can outperform random seeding, but PageRank should not be presented as the single best real-world diffusion strategy. Uniform $p = 0.05$ is a modelling choice, and reply edges measure observed interaction rather than guaranteed influence.


## 12 · Findings and headline numbers for the report


In [ ]:
def _safe(value, ndigits=None):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if ndigits is not None and isinstance(value, (int, float)):
        return round(float(value), ndigits)
    return value


# Reddit corpus shape
n_cascades = int(len(cascades_reddit))
median_size = _safe(cascades_reddit["size"].median())
median_depth = _safe(cascades_reddit["max_depth"].median())
median_t_first = _safe(cascades_reddit["time_to_first_reply_sec"].median(), 1)
median_t_50pct = _safe(cascades_reddit["t_50pct_cumulative_sec"].median(), 1)
median_pct_leaves = _safe(cascades_reddit["pct_leaves"].median(), 3)

# Subreddit defensibility (recompute KW from the cleaned arrays).
from scipy.stats import kruskal as _kw
g_size  = [g["size"].values for _, g in cascades_reddit.groupby("subreddit") if len(g) >= 3]
g_depth = [g["max_depth"].values for _, g in cascades_reddit.groupby("subreddit") if len(g) >= 3]
if g_size and g_depth:
    h_size_v, p_size_v = _kw(*g_size)
    h_depth_v, p_depth_v = _kw(*g_depth)
else:
    h_size_v = p_size_v = h_depth_v = p_depth_v = float("nan")

# Sentiment × diffusion: medians per bucket.
by_bucket_dict = {row["sentiment_bucket"]: {k: _safe(row[k], 2)
                                            for k in ["n_cascades", "median_size",
                                                      "median_max_depth",
                                                      "median_t_first_reply_min",
                                                      "median_pct_leaves"]}
                  for _, row in by_bucket.iterrows()
                  if pd.notna(row["sentiment_bucket"])}

# Topic × diffusion: top 3 topics by median cascade size.
top_topics_by_size = (topic_dom_table.head(3)[["dominant_topic_label",
                                                "n_cascades",
                                                "median_size",
                                                "median_max_depth",
                                                "median_pct_negative"]]
                       .to_dict(orient="records"))

# Influence × diffusion: correlations.
infl_corr = {"spearman_rho_size":  _safe(rho_size, 3),
             "spearman_p_size":     _safe(p_size, 4),
             "spearman_rho_depth": _safe(rho_depth, 3),
             "spearman_p_depth":    _safe(p_depth, 4)}

# YouTube engagement summary.
yt_total = int(len(engagement_youtube))
yt_with_reply = int(engagement_youtube["had_any_reply"].sum())
yt_share_with_reply = _safe(engagement_youtube["had_any_reply"].mean() * 100, 2)
yt_median_replies_engaged = _safe(
    engagement_youtube.loc[engagement_youtube["had_any_reply"], "n_replies"].median()
)

# ICM headline: PageRank top-20 expected spread / random top-20 expected spread.
def _icm_get(strategy, k):
    row = icm_results[(icm_results["strategy"] == strategy) & (icm_results["k"] == k)]
    return float(row["mean_spread"].iloc[0]) if not row.empty else float("nan")
icm_summary = {
    "p": ICM_P,
    "monte_carlo_trials": ICM_TRIALS,
    "pagerank_k20_mean_spread": _safe(_icm_get("PageRank top-k", 20), 1),
    "betweenness_k20_mean_spread": _safe(_icm_get("Betweenness top-k", 20), 1),
    "random_k20_mean_spread": _safe(_icm_get("Random", 20), 1),
    "ratio_pagerank_over_random_k20": _safe(
        _icm_get("PageRank top-k", 20) / _icm_get("Random", 20)
        if _icm_get("Random", 20) else float("nan"), 2),
}

# Sentiment drift.
drift_mean = _safe(drift["drift"].mean() if not drift.empty else None, 3)
drift_median = _safe(drift["drift"].median() if not drift.empty else None, 3)

findings = {
    "generated_at_utc": pd.Timestamp.now(tz="UTC").replace(microsecond=0).isoformat(),
    "reddit": {
        "n_cascades": n_cascades,
        "median_size": median_size,
        "median_max_depth": median_depth,
        "median_time_to_first_reply_sec": median_t_first,
        "median_t_50pct_cumulative_sec": median_t_50pct,
        "median_pct_leaves": median_pct_leaves,
        "kruskal_wallis": {
            "size_h": _safe(h_size_v, 2), "size_p": _safe(p_size_v, 4),
            "depth_h": _safe(h_depth_v, 2), "depth_p": _safe(p_depth_v, 4),
        },
        "sentiment_bucket_medians": by_bucket_dict,
        "sentiment_drift_mean":   drift_mean,
        "sentiment_drift_median": drift_median,
        "top_topics_by_median_size": top_topics_by_size,
        "influence_correlations": infl_corr,
        "icm": icm_summary,
    },
    "youtube": {
        "n_top_level_comments": yt_total,
        "n_with_any_reply": yt_with_reply,
        "share_with_any_reply_pct": yt_share_with_reply,
        "median_replies_among_engaged": yt_median_replies_engaged,
        "n_eligible_channels": int(len(by_channel_min50)),
        "median_reply_rate_pct_eligible_channels": _safe(by_channel_min50["reply_rate_pct"].median(), 1),
    },
    "outputs": {
        "reddit_cascade_metrics_csv":         str((DIFFUSION / "reddit_cascade_metrics.csv").relative_to(PROJECT_ROOT)),
        "youtube_engagement_metrics_csv":     str((DIFFUSION / "youtube_engagement_metrics.csv").relative_to(PROJECT_ROOT)),
        "reddit_diffusion_by_subreddit_csv":  str((DIFFUSION / "reddit_diffusion_by_subreddit.csv").relative_to(PROJECT_ROOT)),
        "reddit_diffusion_by_topic_csv":      str((DIFFUSION / "reddit_diffusion_by_topic.csv").relative_to(PROJECT_ROOT)),
        "reddit_diffusion_by_sentiment_csv":  str((DIFFUSION / "reddit_diffusion_by_sentiment_bucket.csv").relative_to(PROJECT_ROOT)),
        "reddit_cross_cascade_authors_csv":   str((DIFFUSION / "reddit_cross_cascade_authors.csv").relative_to(PROJECT_ROOT)),
        "icm_expected_spread_csv":            str((DIFFUSION / "icm_expected_spread.csv").relative_to(PROJECT_ROOT)),
        "youtube_engagement_by_channel_csv":  str((DIFFUSION / "youtube_engagement_by_channel.csv").relative_to(PROJECT_ROOT)),
        "youtube_engagement_by_video_csv":    str((DIFFUSION / "youtube_engagement_by_video.csv").relative_to(PROJECT_ROOT)),
        "reddit_cascade_growth_curves_csv":   str((DIFFUSION / "reddit_cascade_growth_curves.csv").relative_to(PROJECT_ROOT)),
    },
}

with open(DIFFUSION / "findings_diffusion_summary.json", "w") as f:
    json.dump(findings, f, indent=2, default=str)

print("Findings summary written to findings_diffusion_summary.json:\n")
print(json.dumps(findings, indent=2, default=str))


## Generated artifacts

Relevant existing outputs from this notebook are:

- `data/processed/05_diffusion/reddit_cascade_metrics.csv`
- `data/processed/05_diffusion/youtube_engagement_metrics.csv`
- `data/processed/05_diffusion/reddit_diffusion_by_sentiment_bucket.csv`
- `data/processed/05_diffusion/reddit_diffusion_by_topic.csv`
- `data/processed/05_diffusion/icm_expected_spread.csv`
- `plots/reddit_cascade_size_depth.png`
- `plots/reddit_diffusion_by_topic.png`
- `plots/reddit_influence_vs_diffusion.png`
- `plots/youtube_engagement_sketch.png`
- `plots/icm_expected_spread.png`

## Report-ready findings

- Diffusion uses platform-correct units: Reddit submission cascades and YouTube top-level comment engagement.
- Reddit supports cascade analysis: **55** submission cascades, median size **305** comments, and median depth **9**.
- YouTube is shallow engagement rather than a comparable cascade network: only **12.36%** of top-level comments receive any reply.
- Topic and influence diagnostics are interpreted cautiously: topic shapes cascade size/depth, and centrality is associated with spread, but neither result is causal.
